# Pre-training Model Evaluation: How Do You Measure Quality?

In this notebook, we'll line up multiple **base (pre-trained)** models side by side and apply the core techniques used to measure how "well" a model has been trained.

### Usage
Add as many models as you want to the `MODELS` and `TOKENIZERS` dicts — every test runs across all of them automatically.

### Layers Covered

| # | Layer | Sections |
|---|-------|----------|
| 1 | **Loss-Based Metrics** | PPL, BPC, Token Entropy, Sliding Window PPL |
| 2 | **Benchmark (Log-Likelihood)** | Multiple Choice Ranking |
| 3 | **Linguistic Competence** | Subject-Verb Agreement, Word Order, Turkish Morphology |
| 4 | **Factual Knowledge** | Popular vs Rare, Contrastive Margin |
| 5 | **Reasoning & ICL** | Few-shot ICL Curve, Flipped Labels |
| 6 | **Generation Quality & Calibration** | Rep-4, Distinct-2, ECE, Paraphrase Consistency |
| 7 | **Mechanistic Interpretability** | Logit Lens, Attention Map, Top-K Confidence |

---

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

def clear_gpu():
    """Clear GPU memory after each method."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_device(model):
    """Return the model's device — works for models sharded with device_map=auto too."""
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device(device)

print(f"Device: {device}")


In [ ]:
# ============================================================
# Model loading — Base vs Fine-tuned comparison
# How much has changed on top of the same base (Kumru-2B)?
# ============================================================

model_configs = {
    #"Kumru-2B (base)": {"id": "vngrs-ai/Kumru-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    #"Kara-Kumru (alican)": {"id": "AlicanKiraz0/Kara-Kumru-v1.0-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    #"gemma4-2b": {"id":"google/gemma-4-E2B","kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    "gemma4-2b-it": {"id":"google/gemma-4-E2B-it","kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}}
}

MODELS = {}
TOKENIZERS = {}

for name, cfg in model_configs.items():
    print(f"Loading: {name} ({cfg['id']})...")
    TOKENIZERS[name] = AutoTokenizer.from_pretrained(cfg["id"])
    m = AutoModelForCausalLM.from_pretrained(cfg["id"], **cfg["kwargs"]);
    MODELS[name] = m.to(device) if "device_map" not in cfg["kwargs"] else m
    clear_gpu()

print(f"\n{len(MODELS)} models loaded: {list(MODELS.keys())}")

## Helper functions

Core utilities reused throughout the notebook.

In [ ]:
def get_logprob(model, tokenizer, prefix, continuation):
    """Total log-probability of `continuation` given `prefix`.
    Many tests (agreement, factual, ICL) lean on this."""
    full = prefix + continuation
    inputs = tokenizer(full, return_tensors="pt").to(get_device(model))
    prefix_len = tokenizer(prefix, return_tensors="pt")["input_ids"].shape[1]
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)
    ids = inputs["input_ids"][0]
    return sum(log_probs[i-1, ids[i]].item() for i in range(prefix_len, len(ids)))


def greedy_complete(model, tokenizer, prompt, max_tokens=20):
    """Greedy decoding — pick the highest-probability token at each step."""
    inputs = tokenizer(prompt, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


def sampled_generate(model, tokenizer, prompt, max_tokens=200):
    """Sampled generation — used for diversity metrics."""
    inputs = tokenizer(prompt, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             repetition_penalty=1.1)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


print("Helper functions loaded: get_logprob, greedy_complete, sampled_generate")

## 1. Perplexity (PPL) and BPC (Bits Per Character)

### What is it?
- **Perplexity (PPL):** A metric describing how "uncertain" the model is on average when predicting the next token — effectively how many alternatives it is hesitating between. Mathematically it is the exponential of the cross-entropy loss: `PPL = exp(loss)`.
- **BPC (Bits Per Character):** Converts the loss into bits per character. This is tokenizer-agnostic, which lets you compare models with different vocabulary sizes fairly.

$$BPC = \frac{Loss \times \text{Token Count}}{\text{Character Count} \times \ln(2)}$$

### What do we expect to find?
- A well-trained model should be able to "compress" the language well → **low PPL and BPC**.
- Models trained heavily on Turkish should produce lower PPL on Turkish text.
- Very large PPL (>100) indicates the model has not learned this language/domain well.

### How do we interpret it?
| Metric | Good | Bad |
|--------|------|-----|
| PPL | Low (e.g. 5-30) | High (e.g. 100+) |
| BPC | < 1.5 | > 3.0 |

> **Caveat:** Comparing PPL directly across models with different tokenizers can be misleading (tokenizer efficiency is a confound). BPC removes this issue.

In [ ]:
device

In [ ]:
def advanced_eval(model, tokenizer, text, label):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    input_ids = inputs["input_ids"]
    char_count = len(text)
    token_count = input_ids.shape[1]

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss.item()
        ppl = np.exp(loss)
        bpc = (loss * token_count) / (char_count * np.log(2))

    return {"Model": label, "Loss": round(loss, 4), "PPL": round(ppl, 4), "BPC": round(bpc, 4)}

test_texts = [
    "Merkez Bankası faiz kararı piyasalarda dalgalanmaya neden oldu.",
    "Borsa İstanbul günü yüzde iki artışla kapattı.",
    "Enflasyon oranı beklentilerin üzerinde açıklandı.",
]

for text in test_texts:
    results = [advanced_eval(MODELS[name], TOKENIZERS[name], text, name) for name in MODELS]
    print(f"\nText: {text[:60]}...")
    display(pd.DataFrame(results))
clear_gpu()

## 2. Token Entropy and Confidence Analysis

### What is it?
For every token we measure two things:
- **Confidence:** How much probability the model assigned to the actual (correct) token. 90% means very confident, 2% is barely better than guessing.
- **Entropy:** How "spread out" the model's probability distribution is. Measured via Shannon entropy. Low entropy = model focused on a few tokens (confident). High entropy = probability scattered across many tokens (uncertain).

### What do we expect to find?
- **Function words** (de, ve, bir, için): high confidence, low entropy → easy to predict from context.
- **Content words** (Ankara, technology): lower confidence, higher entropy → many alternatives are possible.
- A good model should be confident where things are predictable and honestly uncertain where they aren't.

### How do we interpret it?
- A token with **high confidence + low entropy** → the model has learned that context well.
- **Low confidence + high entropy** → uncertain; either the context is insufficient or the model never learned that pattern.
- Persistently low confidence on every token → the model is generally undertrained.

In [ ]:
def analyze_token_confidence(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        logits = model(input_ids).logits[0, :-1, :]
        probs = F.softmax(logits, dim=-1)
        target_ids = input_ids[0, 1:]
        target_probs = probs[torch.arange(len(target_ids)), target_ids]
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)

    #return entropy,target_probs
    return pd.DataFrame({
        "Token": [tokenizer.decode([t]) for t in target_ids],
        "Confidence (%)": (target_probs * 100).cpu().float().numpy().round(2),
        "Entropy": entropy.cpu().float().numpy().round(4)
  })

In [ ]:
text = "Merkez Bankası faiz oranını yüzde elli olarak belirledi."
for name in MODELS:
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    display(analyze_token_confidence(MODELS[name], TOKENIZERS[name], text))
clear_gpu()

## 3. Logit Lens Visualization

### What is it?
Logit Lens is a technique that peeks at the model's **intermediate layers** to see how information is built up step by step. Normally we only see the final layer's output; with the logit lens we run each layer's hidden state through the final norm + lm_head and ask "what would this layer have predicted on its own?".

We provide two views:
- **Plot (`plot_logit_lens`):** Shows how the probability of a target word (e.g. "Ankara") rises across layers.
- **Table (`robust_logit_lens`):** Lists the model's top prediction every N layers. Architecture-agnostic (Qwen / Llama / GPT-2).

### What do we expect to find?
- **Early layers:** target word should be low probability — the model is still processing surface features (n-grams, position).
- **Middle layers:** probability should start rising — syntax and semantic relations are being built.
- **Final layers:** the correct answer's probability should be markedly high — knowledge has crystallized.

### How do we interpret it?
- **Sharp sigmoid-shaped rise:** the model learned the information within a specific layer band → healthy.
- **Early rise:** the model can resolve this even in lower layers → especially strong learning.
- **Flat line (no rise):** the model never learned this information.
- **Cross-model comparison:** which model resolves the information at earlier layers? That model has internalized the fact more "deeply".

In [ ]:
def get_final_norm(model):
    """Dynamically locate the model's final norm layer (Qwen / Llama vs GPT-2)."""
    if hasattr(model, 'model') and hasattr(model.model, 'norm'):
        return model.model.norm
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'ln_f'):
        return model.transformer.ln_f
    return None

def plot_logit_lens(model, tokenizer, text, target_word=" Ankara"):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    target_id = tokenizer.encode(target_word)[-1]
    final_norm = get_final_norm(model)

    with torch.no_grad():
        outputs = model(inputs["input_ids"], output_hidden_states=True)
        hidden_states = outputs.hidden_states

    probs_list = []
    for h in hidden_states:
        norm_h = final_norm(h) if final_norm is not None else h
        logits = model.lm_head(norm_h)[0, -1, :]
        probs_list.append(F.softmax(logits, dim=-1)[target_id].item())

    plt.figure(figsize=(10, 4))
    plt.plot(probs_list, marker='o', color='teal', linewidth=2)
    plt.title(f"Probability of '{target_word}' across layers")
    plt.xlabel("Layer")
    plt.ylabel("Probability")
    plt.grid(True, alpha=0.3)
    plt.show()

def robust_logit_lens(model, tokenizer, text, layer_step=4):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    final_norm = get_final_norm(model)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states

    results = []
    for i in range(0, len(hidden_states), layer_step):
        state = final_norm(hidden_states[i]) if final_norm is not None else hidden_states[i]
        logits = model.lm_head(state)
        top_token = tokenizer.decode([torch.argmax(logits[0, -1, :]).item()])
        results.append({"Layer": i, "Predicted Next Token": top_token})
    return pd.DataFrame(results)

In [ ]:
text = "Türkiye'nin merkez bankasının adı "
for name in MODELS:
    print(f"\n--- {name}: Logit Lens (plot) ---")
    plot_logit_lens(MODELS[name], TOKENIZERS[name], text, " TCMB")
    print(f"\n--- {name}: Robust Logit Lens (table) ---")
    display(robust_logit_lens(MODELS[name], TOKENIZERS[name], text))
    clear_gpu()

## 4. Logical Surprisal Analysis

### What is it?
Surprisal measures how "surprised" the model is when it sees a token: `-log P(token | context)`. Here we use it as a **logical-consistency test**: we feed the same context with a sensible and a nonsensical continuation and compare which surprises the model more.

**Example:**
- Context: *"Hava çok sıcak olduğu için..."* ("Because it's very hot...")
- Logical continuation: *"su aldım."* ("I bought water.") → low surprisal expected
- Illogical continuation: *"ateş aldım."* ("I bought fire.") → high surprisal expected

### What do we expect to find?
- A good model should assign **noticeably higher** loss to the illogical continuation.
- The "Ratio (x)" column shows this multiplier: 1.5x+ is good, 2x+ is a strong separation.
- Ratio close to 1.0 means the model can't distinguish logical from illogical → weak world knowledge.

### How do we interpret it?
- **High ratio (>1.5x):** the model has grasped the cause-effect relation.
- **Low ratio (~1.0x):** the model finds the two continuations equally plausible → world knowledge for this context is missing.
- **Across models:** whichever model produces the larger ratio has learned real-world relations more accurately.

In [ ]:
def compare_surprisal(model, tokenizer, context, logic_word, illogic_word, model_name=""):
    def get_last_loss(text):
        inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
        with torch.no_grad():
            outputs = model(inputs["input_ids"], labels=inputs["input_ids"])
            logits, labels = outputs.logits[0, :-1, :], inputs["input_ids"][0, 1:]
            return F.cross_entropy(logits[-1:], labels[-1:]).item()

    l_ok, l_bad = get_last_loss(context + logic_word), get_last_loss(context + illogic_word)
    return {
        "Model": model_name,
        "Logical": f"{logic_word} ({l_ok:.4f})",
        "Illogical": f"{illogic_word} ({l_bad:.4f})",
        "Ratio (x)": round(l_bad / l_ok, 2)
    }

In [ ]:
context = "Enflasyon çok yükseldiği için merkez bankası faizi "
results = [
    compare_surprisal(MODELS[name], TOKENIZERS[name], context, " artırdı.", " düşürdü.", name)
    for name in MODELS
]
display(pd.DataFrame(results))
clear_gpu()

## 5. Multiple Choice (Ranking) Test

### What is it?
We give the model a question and several options, then compute the **log-probability score** for each option. The option with the highest score is the model's "most likely" answer. This is the standard way to evaluate base models on benchmarks (MMLU, HellaSwag, ARC, etc.).

**How it works:**
1. `prompt + option` are concatenated
2. Log-probabilities of the option's tokens are summed
3. The option with the highest total score "wins"

### What do we expect to find?
- The correct answer (e.g. "Paris") should have the highest score.
- The **gap** between scores matters: a large gap means the model is very confident, a small one means it's hesitant.
- A wrong answer ranked first means the model has not learned the fact.

### How do we interpret it?
- **Correct answer ranked 1st with a large score gap:** the model has solidly learned this fact.
- **Correct answer ranked 1st but small gap:** the model knows it but isn't fully confident.
- **Wrong answer ranked 1st:** the model has either learned the wrong information or never learned it at all.
- **Across models:** on the same question, which model picks the correct answer with the larger margin?

In [ ]:
def rank_choices(model, tokenizer, prompt, choices):
    """Compute log-prob for each option and rank them.
    Uses prefix_len — safe against subword boundary shift."""
    results = []
    prefix_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    for choice in choices:
        full_text = prompt + choice
        inputs = tokenizer(full_text, return_tensors="pt").to(get_device(model))
        ids = inputs["input_ids"][0]
        n_choice_tokens = len(ids) - prefix_len

        with torch.no_grad():
            logits = model(inputs["input_ids"]).logits[0]
            log_probs = F.log_softmax(logits, dim=-1)

            total_lp = sum(log_probs[i-1, ids[i]].item() for i in range(prefix_len, len(ids)))
            norm_lp = total_lp / n_choice_tokens if n_choice_tokens > 0 else total_lp

        results.append({"Choice": choice, "Score (raw)": round(total_lp, 4),
                        "Score (norm)": round(norm_lp, 4), "Tokens": n_choice_tokens})
    return pd.DataFrame(results).sort_values("Score (norm)", ascending=False)

In [ ]:
ranking_tests = [
    ("Türkiye'nin merkez bankasının adı nedir? Cevap:", [" TCMB", " FED", " ECB"], "TCMB"),
    ("Borsa İstanbul'un kısaltması:", [" BIST", " NYSE", " DAX"], "BIST"),
    ("Enflasyon artınca merkez bankası genellikle faizi", [" artırır.", " düşürür.", " sabit tutar."], "artırır."),
    ("Bir şirketin toplam varlıklarından borçları çıkarılınca kalan değere", [" özsermaye", " gelir", " gider"], "özsermaye"),
]

for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    for prompt, choices, correct in ranking_tests:
        print(f"\n  Question: {prompt} (Correct: {correct})")
        display(rank_choices(MODELS[name], TOKENIZERS[name], prompt, choices))
clear_gpu()

## 6. Tokenizer Efficiency (Fertility) Analysis

### What is it?
Tokenizer fertility measures how many tokens a piece of text gets split into and the **characters-per-token ratio**. Even with great model weights, a poor tokenizer will seriously hurt performance — burning more tokens for the same meaning means a shorter effective context window and slower inference.

**Characters per token** = total character count / token count

### What do we expect to find?
- Tokenizers trained specifically for Turkish should produce a higher chars/token ratio (more efficient).
- English-heavy tokenizers (e.g. GPT-2) tokenize Turkish into many pieces — especially splitting suffixes (`lar`, `ler`, `daki`, `ndaki`) into separate tokens.
- The `First 10 Tokens` column tells you whether the splits are meaningful subwords or just single characters — a quality signal.

### How do we interpret it?
| Chars/Token | Verdict |
|-------------|---------|
| > 4.0 | Very efficient — well aligned with Turkish |
| 2.5 - 4.0 | Average — general-purpose tokenizer |
| < 2.5 | Inefficient — fragments Turkish text heavily |

> **Note:** Tokenizer efficiency directly affects PPL comparisons. An inefficient tokenizer artificially lowers PPL (since each token carries less information). BPC normalizes for this.

In [ ]:
def tokenizer_fertility_analysis(text, tokenizers_dict):
    results = []
    char_length = len(text)
    for name, tokenizer in tokenizers_dict.items():
        tokens = tokenizer.encode(text)
        token_count = len(tokens)
        chars_per_token = char_length / token_count if token_count > 0 else 0
        results.append({
            "Model": name,
            "Token Count": token_count,
            "Chars per Token": round(chars_per_token, 2),
            "First 10 Tokens": [tokenizer.decode([t]) for t in tokens][:10]
        })
    return pd.DataFrame(results)

In [ ]:
long_text = "Merkez Bankası'nın faiz politikası, enflasyon beklentileri ve döviz kuru hareketleri piyasalardaki yatırımcı davranışını doğrudan etkilemektedir."
display(tokenizer_fertility_analysis(long_text, TOKENIZERS))

## 7. Shannon Entropy and Top-K Confidence Distribution

### What is it?
In Section 2 we looked at entropy and confidence as a table. Here we examine the same concept **visually**: we plot the K tokens to which the model assigns the highest probability at the final position as a bar chart. The Shannon entropy is shown in the title.

This is like "looking inside the model's head" — it shows the decision mechanism at a single prediction step.

### What do we expect to find?
- **Confident model:** 1-2 tokens with very high probability, the rest near zero. Low entropy.
- **Uncertain model:** probability spread across many tokens. High entropy.
- For very deterministic contexts (e.g. "Türkiye'nin Başkenti:") we expect a good model to show a single dominant peak.

### How do we interpret it?
- **One dominant bar:** model is very confident → strong learning in this context.
- **Several close bars:** model is hesitating between alternatives.
- **Flat distribution:** model is essentially guessing → very weak.
- **Across models:** in the same context, which model produces a more "peaked" (confident) distribution?

In [ ]:
def plot_top_k_confidence(model, tokenizer, text, k=5, model_name="Model"):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()
    vals, idxs = torch.topk(probs, k)

    plt.figure(figsize=(10, 4))
    sns.barplot(x=vals.cpu().float().numpy()*100, y=[tokenizer.decode([i]) for i in idxs], palette="viridis")
    plt.title(f"{model_name} - Top {k} Confidence | Entropy: {entropy:.4f}\nContext: '{text}'")
    plt.xlabel("Probability (%)")
    plt.show()




In [ ]:
text = "Borsa İstanbul'un kısaltması : "
for name in MODELS:
    plot_top_k_confidence(MODELS[name], TOKENIZERS[name], text, k=10, model_name=name)
    clear_gpu()

## 8. Real Perplexity with a Sliding Window

### What is it?
The PPL from Section 1 works fine for short text, but on long documents we hit the model's **context-window limit**. The sliding window method scans the text in overlapping windows shifted by `stride` and aggregates the per-window losses to compute a **true PPL across the entire document**.

**How it works:**
1. The text is broken into windows of size `max_length`.
2. Each window slides by `stride` (with overlap).
3. Only the loss for newly seen tokens is counted (the previous overlap is masked with `-100`).
4. All losses are averaged and `exp()` is applied → true PPL.

### What do we expect to find?
- Sliding window PPL should be close to short-text PPL but slightly different — it has more context to draw on.
- Good model: even on highly repetitive text (the same sentence 10 times) PPL stays low → it has captured the pattern.
- Bad model: PPL stays high because it can't hold information as the context grows.

### How do we interpret it?
- Compare with the Section 1 PPL: a large discrepancy can indicate weak long-context handling.
- Cross-model differences highlight which model stays more consistent in long-text scenarios.

In [ ]:
def calculate_sliding_window_ppl(model, tokenizer, text, stride=512, max_length=1024):
    encodings = tokenizer(text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)
    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(get_device(model))
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            nlls.append(outputs.loss)
        prev_end_loc = end_loc
        if end_loc == seq_len: break
    return torch.exp(torch.stack(nlls).mean()).item()




In [ ]:
long_test_text = " " + " ".join(["Finansal piyasalarda volatilite artmaya devam ediyor."] * 10)

results = []
for name in MODELS:
    ppl = calculate_sliding_window_ppl(MODELS[name], TOKENIZERS[name], long_test_text)
    results.append({"Model": name, "Sliding Window PPL": round(ppl, 4)})
    clear_gpu()
display(pd.DataFrame(results))

## 9. Attention Map (Heatmap) Visualization

### What is it?
The attention map visualizes the transformer's **self-attention mechanism**. It plots, as a heatmap, how much each token "attends to" every other token. Rows are the "attending" tokens; columns are the "attended" tokens. Bright cells mean strong attention.

Here we show the **first head of the last layer** — usually where the most "semantic" attention patterns live.

### What do we expect to find?
- **Diagonal:** every token attends to itself and the previous token → the basic autoregressive pattern.
- **Vertical stripes:** many tokens attend to a particular token → that's an "important" word (e.g. subject, verb).
- **Block structures:** nearby tokens attend to each other in groups → syntactic structure (clause boundaries).

### How do we interpret it?
- **Heavy attention on "Ankara":** the model is resolving the capital-city fact through this token.
- **Attention to the BOS / first token:** common in many models — used as an "information sink" (attention sink).
- **Diffuse attention:** the head hasn't learned a clear pattern at this layer.
- **Across models:** comparing attention patterns on the same text shows which model has developed more structured attention.

> **Note:** Only models loaded with `attn_implementation="eager"` return attention outputs. Models using SDPA may not support this.

In [ ]:
def plot_attention_map(model, tokenizer, text, layer=-1, head=0, model_name=""):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    tokens = [tokenizer.decode([t]) for t in inputs["input_ids"][0]]

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions
    if attentions is None:
        print(f"{model_name}: did not return attention (output_attentions may not be supported).")
        return

    valid_attns = [a for a in attentions if a is not None]
    if not valid_attns:
        print(f"{model_name}: no Full Attention layers available to visualize.")
        return

    if abs(layer) >= len(valid_attns):
        print(f"{model_name}: invalid layer! Only {len(valid_attns)} Full Attention layers exist.")
        return

    attn_matrix = valid_attns[layer][0, head, :, :].float().cpu().numpy()

    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis", square=True)
    plt.title(f"{model_name} - Attention Map (Layer {layer}, Head {head})")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
text = "Merkez Bankası faiz oranını artırdı."
for name in MODELS:
    plot_attention_map(MODELS[name], TOKENIZERS[name], text, layer=-1, head=0, model_name=name)
    clear_gpu()

---

# Layer 3: Linguistic Competence

---

## 10. Subject-Verb Agreement Test

### What is it?
Has the model learned **hierarchical syntax**? The classic test: place "attractor" (distractor) nouns between the subject and the verb and check whether the model still picks the correct number agreement (singular/plural).

**Example:**
- *"The keys to the cabinet..."* → `are` (correct, subject "keys" is plural) vs `is` (wrong — "cabinet" is singular but isn't the subject)
- If the model is fooled by "cabinet" → it's using a surface proximity heuristic and hasn't really learned syntax.

**Refs:** Linzen et al. (2016), Goldberg (2019), Marvin & Linzen (2018)

### What do we expect to find?
- **Easy (1 attractor):** any 1B+ model should pass 90%+.
- **Hard (center-embedding):** only strong models clear 50%+. Failure on a small model is normal.
- The difficulty breakdown matters a lot: Easy good + Hard bad = surface pattern matching.

### How do we interpret it?
| Outcome | Verdict |
|---------|---------|
| Easy ✅ Hard ✅ | Strong hierarchical syntax |
| Easy ✅ Hard ❌ | Surface patterns OK, deep structure weak |
| Easy ❌ | Basic-level problem — model size / data insufficient |

In [ ]:
def run_agreement_tests(model, tokenizer, model_name=""):
    """Subject-verb agreement tests — finance domain, Turkish."""
    tests = [
        ("Bankanın şubelerindeki müdürler", " toplantıya katıldı.", " toplantıya katıldım.",
         "easy", "3rd-person plural agreement"),
        ("Yatırımcıların portföyündeki hisse senetleri", " değer kazandı.", " değer kazandım.",
         "easy", "Attractor 'portföy' singular, subject 'senetleri' plural"),
        ("Şirketin denetçilerinin hazırladığı rapor", " onaylandı.", " onaylandılar.",
         "medium", "Nested possessive — subject 'rapor' is singular"),
        ("Fonun yöneticisinin danıştığı analistler", " uyardı.", " uyardım.",
         "hard", "Two levels of nested possessives"),
        ("Merkez bankasının açıkladığı verilere göre piyasalar", " düştü.", " düştüm.",
         "hard", "Long-distance dependency — subject 'piyasalar'"),
    ]

    results = []
    for prefix, correct, wrong, diff, note in tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        passed = lp_c > lp_w
        results.append({
            "Prefix": prefix[:40] + "...",
            "Difficulty": diff,
            "Passed": "PASS" if passed else "FAIL",
            "Margin": round(lp_c - lp_w, 3),
            "Note": note
        })

    df = pd.DataFrame(results)
    total = sum(1 for r in results if r["Passed"] == "PASS")
    print(f"  {model_name}: {total}/{len(results)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_agreement_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

## 11. Word Order Test

### What is it?
A grammatically correct sentence is compared against a version where the same words are in random order. If the model has learned that word order matters, it should assign **markedly lower PPL** to the correct sentence.

### What do we expect to find?
- PPL ratio (scrambled / correct) > 5x: very sensitive — good.
- 2-5x: reasonable.
- < 2x: weak knowledge of word order.

In [ ]:
def run_word_order_tests(model, tokenizer, model_name=""):
    pairs = [
        ("Merkez Bankası faiz oranını artırdı.", "Artırdı oranını faiz Bankası Merkez."),
        ("Yatırımcılar borsadan hisse senedi aldı.", "Aldı senedi hisse borsadan yatırımcılar."),
        ("Şirket bilançosunu kamuoyuyla paylaştı.", "Paylaştı kamuoyuyla bilançosunu şirket."),
        ("Dolar kuru dün akşam rekor kırdı.", "Kırdı rekor akşam dün kuru dolar."),
    ]

    results = []
    for good, bad in pairs:
        inputs_g = tokenizer(good, return_tensors="pt").to(get_device(model))
        inputs_b = tokenizer(bad, return_tensors="pt").to(get_device(model))
        with torch.no_grad():
            loss_g = model(**inputs_g, labels=inputs_g["input_ids"]).loss.item()
            loss_b = model(**inputs_b, labels=inputs_b["input_ids"]).loss.item()
        ppl_g, ppl_b = math.exp(loss_g), math.exp(loss_b)
        ratio = ppl_b / ppl_g if ppl_g > 0 else 0
        results.append({
            "Correct": good, "Scrambled": bad,
            "PPL (correct)": round(ppl_g, 1), "PPL (scrambled)": round(ppl_b, 1),
            "Ratio": round(ratio, 1), "Passed": "PASS" if ppl_g < ppl_b else "FAIL"
        })
    return pd.DataFrame(results)

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_word_order_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

## 12. Turkish Morphology: Vowel Harmony and Person Agreement

### What is it?
Turkish is an agglutinative language. We test two core morphological rules:

**Vowel Harmony:** the vowel in a suffix matches the last vowel of the stem.
- Back vowels (a, ı, o, u) → back suffix: *kitap**lar*** (correct), *kitap**ler*** (wrong)
- Front vowels (e, i, ö, ü) → front suffix: *ev**ler*** (correct), *ev**lar*** (wrong)

**Person Agreement:** verb conjugation must agree with the subject.
- *Ali okula gitti.* (3rd singular) vs *Ali okula gittim.* (1st singular — wrong)

**Refs:** Oflazer (1994), Rust et al. (2021) "How Good Is Your Tokenizer?"

### What do we expect to find?
- Models trained heavily on Turkish (Kumru, Turkish-GPT2) should be strong here.
- Multilingual models that are weak on Turkish (Qwen) may fail vowel harmony.
- Tokenizer fertility matters directly: tokenizers that can isolate suffixes as separate tokens have an advantage.

In [ ]:
def run_turkish_morphology_tests(model, tokenizer, model_name=""):
    """Turkish vowel harmony and person agreement tests — finance domain."""

    vowel_tests = [
        ("Faizler", "den", "dan", "Front: e→e (faiz+ler+den)"),
        ("Bankalar", "dan", "den", "Back: a→a (banka+lar+dan)"),
        ("Borsa", "lar", "ler", "Back plural (borsa+lar)"),
        ("Getiri", "ler", "lar", "Front plural (getiri+ler)"),
        ("Portföy", "leri", "ları", "Front possessive (portföy+leri)"),
        ("Yatırım", "ları", "leri", "Back possessive (yatırım+ları)"),
    ]

    case_tests = [
        ("Analist raporu", " yayımladı.", " yayımladım.", "3sg vs 1sg"),
        ("Yatırımcılar hisse", " sattı.", " sattın.", "3pl vs 2sg"),
        ("Dün akşam piyasalar", " kapandı.", " kapanacak.", "Temporal: dün → past tense"),
        ("Ben bu hisseyi", " aldım.", " aldı.", "1sg vs 3sg"),
    ]

    results = []

    for prefix, correct, wrong, note in vowel_tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        results.append({
            "Test": "Vowel Harmony", "Prefix": prefix,
            "Correct": correct, "Wrong": wrong,
            "Passed": "PASS" if lp_c > lp_w else "FAIL",
            "Margin": round(lp_c - lp_w, 3), "Note": note
        })

    for prefix, correct, wrong, note in case_tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        results.append({
            "Test": "Person Agreement", "Prefix": prefix,
            "Correct": correct, "Wrong": wrong,
            "Passed": "PASS" if lp_c > lp_w else "FAIL",
            "Margin": round(lp_c - lp_w, 3), "Note": note
        })

    df = pd.DataFrame(results)
    vowel_pass = sum(1 for r in results if r["Test"] == "Vowel Harmony" and r["Passed"] == "PASS")
    case_pass = sum(1 for r in results if r["Test"] == "Person Agreement" and r["Passed"] == "PASS")
    print(f"  {model_name}: Vowel Harmony {vowel_pass}/{len(vowel_tests)}, Person Agreement {case_pass}/{len(case_tests)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_turkish_morphology_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

---

# Layer 4: Factual Knowledge

---

## 13. Factual Knowledge: Popular vs Rare

### What is it?
How much **world knowledge** has the model encoded into its parameters? We test on two levels:

- **Popular (frequent) facts:** "The capital of France is Paris" — appears very often in training data, almost any model knows it.
- **Rare facts:** "The capital of Burkina Faso is Ouagadougou" — appears very rarely, only well-trained large models know it.

**Contrastive Margin:** the log-prob difference between the correct and wrong answers. Margin > 0 = the model knows the right answer. Larger margin = more confident.

**Refs:** Petroni et al. (2019) LAMA, Chang et al. (2024) learnability threshold, Dai et al. (2022) Knowledge Neurons

### What do we expect to find?
- All models should succeed on popular facts.
- Only large, well-trained models should succeed on rare facts.
- Chang et al. (2024) finding: "unpopular" facts that fall below the learnability threshold are **never learned**.
- Contrastive margin is more informative than a binary correct/wrong — it shows how confident the model is.

### How do we interpret it?
| Popular | Rare | Verdict |
|---------|------|---------|
| ✅ High margin | ✅ | Strong knowledge capacity |
| ✅ High margin | ❌ | Normal — model size / data limited |
| ❌ | ❌ | Serious problem — coverage or training is insufficient |
| ❌ | ✅ | Strange — likely data contamination |

In [ ]:
def contrastive_margin(model, tokenizer, prompt, correct, incorrect):
    """Log-prob gap between the correct and wrong answer."""
    lp_c = get_logprob(model, tokenizer, prompt, correct)
    lp_w = get_logprob(model, tokenizer, prompt, incorrect)
    return {"margin": round(lp_c - lp_w, 3), "passed": lp_c > lp_w}


def run_factual_tests(model, tokenizer, model_name=""):
    """Popular vs rare financial knowledge test."""
    tests = [
        ("Türkiye'nin merkez bankasının kısaltması", " TCMB", " FED", "popular"),
        ("Borsa İstanbul'un kısaltması", " BIST", " NYSE", "popular"),
        ("Türkiye'nin para birimi", " Türk lirası", " dolar", "popular"),
        ("Enflasyon hesaplanırken kullanılan endeksin adı", " TÜFE", " BİST", "popular"),
        ("Şirketlerin finansal durumunu gösteren tabloya", " bilanço", " fatura", "medium"),
        ("Tahvil fiyatı yükseldiğinde getirisi", " düşer", " artar", "medium"),
        ("Black-Scholes modeli hangi finansal enstrümanın fiyatlamasında kullanılır?", " Opsiyon", " Tahvil", "rare"),
        ("Basel III düzenlemesinde bankaların asgari sermaye yeterlilik oranı", " yüzde sekiz", " yüzde iki", "rare"),
    ]

    results = []
    for prompt, correct, incorrect, diff in tests:
        r = contrastive_margin(model, tokenizer, prompt, correct, incorrect)
        results.append({
            "Prompt": prompt[:45], "Correct": correct, "Wrong": incorrect,
            "Difficulty": diff, "Margin": r["margin"],
            "Passed": "PASS" if r["passed"] else "FAIL"
        })

    df = pd.DataFrame(results)
    for diff in ["popular", "medium", "rare"]:
        subset = [r for r in results if r["Difficulty"] == diff]
        passed = sum(1 for r in subset if r["Passed"] == "PASS")
        print(f"  {model_name} [{diff}]: {passed}/{len(subset)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_factual_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

---

# Layer 5: Reasoning & In-Context Learning

---

## 14. ICL (In-Context Learning) Curve

### What is it?
One of the most important emergent abilities of base models: extracting a pattern from in-prompt examples and applying it. The **ICL curve** shows how the loss on a target answer drops as we go from 0-shot to N-shot.

**How it works:**
1. 0-shot: just the question → high loss
2. 1-shot: 1 example + the question → loss begins to drop
3. N-shot: N examples + the question → loss should drop with each new example

**Refs:** Brown et al. (2020) GPT-3, Garg et al. (2022)

### What do we expect to find?
- Loss should decrease with each added example — that's ICL working.
- The rate of decrease reflects the model's ICL capacity.
- Small models may decrease slowly or not at all — that's normal.

### How do we interpret it?
- **Sharp drop (1-2 shot):** model picks up the pattern fast → strong ICL.
- **Gradual drop:** learning, but slowly.
- **Flat line:** ICL isn't working — the model can't learn from examples.

In [ ]:
def evaluate_icl_curve(model, tokenizer, base_prompt, examples,
                       target_question, target_answer, model_name=""):
    """Target loss as we go from 0-shot to N-shot."""
    results = []
    current_prompt = base_prompt

    for k in range(len(examples) + 1):
        if k > 0:
            current_prompt += examples[k-1] + "\n"

        full = current_prompt + target_question + target_answer
        inputs = tokenizer(full, return_tensors="pt").to(get_device(model))
        target_ids = tokenizer.encode(target_answer, add_special_tokens=False)
        target_len = len(target_ids)

        with torch.no_grad():
            logits = model(inputs["input_ids"]).logits[0]
            logits_slice = logits[-(target_len+1):-1, :]
            log_probs = F.log_softmax(logits_slice, dim=-1)
            score = log_probs[torch.arange(target_len),
                              torch.tensor(target_ids).to(get_device(model))].mean().item()

        results.append({"k_shot": k, "target_loss": round(-score, 4)})

    return pd.DataFrame(results)


def plot_icl_curves(models_dict, tokenizers_dict, base_prompt, examples,
                    target_question, target_answer):
    """Show all models' ICL curves on one plot."""
    plt.figure(figsize=(10, 5))

    for name in models_dict:
        df = evaluate_icl_curve(models_dict[name], tokenizers_dict[name],
                                base_prompt, examples, target_question,
                                target_answer, name)
        plt.plot(df["k_shot"], df["target_loss"], marker='o', label=name, linewidth=2)

    plt.xlabel("Number of Examples (k-shot)")
    plt.ylabel("Target Loss (lower = better)")
    plt.title("ICL Learning Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Financial-term translation ICL test
base = "Finansal terimlerin Türkçe karşılıkları:\n"
examples = ["Equity -> Özsermaye", "Bond -> Tahvil", "Inflation -> Enflasyon", "Dividend -> Temettü"]
target_q = "Hedge -> "
target_a = "Korunma"

plot_icl_curves(MODELS, TOKENIZERS, base, examples, target_q, target_a)
clear_gpu()

## 15. Flipped Labels Test (Real ICL or Just Semantic Prior?)

### What is it?
The most critical ICL test. We compare normal labels with inverted labels:

- **Normal:** *"Great! → Positive, Terrible → Negative, Amazing → ?"* → Positive (easy — aligned with semantic prior)
- **Flipped:** *"Great! → Negative, Terrible → Positive, Amazing → ?"* → Negative (hard — labels are inverted)

If the model says "Negative" in the flipped condition → **real ICL** (it's following the in-context pattern).
If it says "Positive" → **semantic prior dominates**, ICL is weak.

**Ref:** Wei et al. (2023) "Larger LMs Do In-Context Learning Differently"

### What do we expect to find?
- **Small models (<3B):** failing the flipped condition is **normal and expected**. Semantic prior dominates.
- **Large models (7B+):** expected to follow the pattern even when flipped.
- This test cleanly demonstrates the effect of model size on ICL capacity.

### How do we interpret it?
| Normal | Flipped | Verdict |
|--------|---------|---------|
| ✅ | ✅ | Real ICL — learning from context |
| ✅ | ❌ | Semantic prior dominates — ICL weak (expected on small models) |
| ❌ | ❌ | ICL not working |

In [ ]:
def test_flipped_labels(model, tokenizer, model_name=""):
    """Normal vs flipped label ICL test — finance sentiment."""

    normal_prompt = "Faiz artırıldı. -> Negatif\nEnflasyon düştü. -> Pozitif\nİşsizlik arttı. ->"
    flipped_prompt = "Faiz artırıldı. -> Pozitif\nEnflasyon düştü. -> Negatif\nİşsizlik arttı. ->"

    lp_pos_normal = get_logprob(model, tokenizer, normal_prompt, " Pozitif")
    lp_neg_normal = get_logprob(model, tokenizer, normal_prompt, " Negatif")
    normal_ok = lp_neg_normal > lp_pos_normal  # "İşsizlik arttı" -> we expect Negatif

    lp_pos_flipped = get_logprob(model, tokenizer, flipped_prompt, " Pozitif")
    lp_neg_flipped = get_logprob(model, tokenizer, flipped_prompt, " Negatif")
    flipped_ok = lp_pos_flipped > lp_neg_flipped  # in flipped we expect Pozitif

    if normal_ok and flipped_ok:
        verdict = "REAL ICL — following the pattern"
    elif normal_ok and not flipped_ok:
        verdict = "SEMANTIC PRIOR dominates — weak ICL (expected on small models)"
    else:
        verdict = "ICL not working"

    return {
        "Model": model_name,
        "Normal": "PASS" if normal_ok else "FAIL",
        "Normal Margin": round(lp_neg_normal - lp_pos_normal, 3),
        "Flipped": "PASS" if flipped_ok else "FAIL",
        "Flipped Margin": round(lp_pos_flipped - lp_neg_flipped, 3),
        "Verdict": verdict
    }

In [ ]:
results = [test_flipped_labels(MODELS[name], TOKENIZERS[name], name) for name in MODELS]
display(pd.DataFrame(results))
clear_gpu()

---

# Layer 6: Generation Quality & Calibration

---

## 16. Generation Quality: Repetition and Diversity

### What is it?
When the model generates long text, how repetitive is it and how diverse? Two core metrics:

- **Rep-4:** 4-gram repetition rate. Measures how often the same 4-word sequence appears in the generated text. `1 - (unique 4-grams / total 4-grams)`.
- **Distinct-2:** Unique 2-gram ratio. High = diverse text, low = monotonous.

**Refs:** Holtzman et al. (2020) "Neural Text Degeneration", Su et al. (2022) SimCTG

### What do we expect to find?
- Good model: low repetition, high diversity.
- Bad model: the same sentence/pattern repeats many times (degeneration).

### How do we interpret it?
| Metric | Good | Warning | Serious problem |
|--------|------|---------|-----------------|
| Rep-4 | < 5% | 5-15% | > 15% |
| Distinct-2 | > 0.8 | 0.5-0.8 | < 0.5 |

In [ ]:
def generation_metrics(text):
    """Compute Rep-4 and Distinct-2."""
    words = text.split()
    n = len(words)
    if n < 4:
        return {"rep_4": 0.0, "distinct_2": 0.0, "word_count": n}
    fours = [tuple(words[i:i+4]) for i in range(n-3)]
    twos = [tuple(words[i:i+2]) for i in range(n-1)]
    return {
        "rep_4": round(1 - len(set(fours)) / len(fours), 4),
        "distinct_2": round(len(set(twos)) / len(twos), 4),
        "word_count": n
    }


def run_generation_quality(models_dict, tokenizers_dict, prompts):
    """Measure generation quality across multiple prompts."""
    results = []
    for name in models_dict:
        all_rep4, all_dist2 = [], []
        for prompt in prompts:
            text = sampled_generate(models_dict[name], tokenizers_dict[name], prompt, max_tokens=150)
            m = generation_metrics(text)
            all_rep4.append(m["rep_4"])
            all_dist2.append(m["distinct_2"])

        avg_rep4 = round(np.mean(all_rep4), 4)
        avg_dist2 = round(np.mean(all_dist2), 4)
        results.append({
            "Model": name,
            "Rep-4 (avg)": avg_rep4,
            "Distinct-2 (avg)": avg_dist2,
            "Rep-4 Status": "OK" if avg_rep4 < 0.05 else ("WARN" if avg_rep4 < 0.15 else "ISSUE"),
            "Dist-2 Status": "OK" if avg_dist2 > 0.8 else ("WARN" if avg_dist2 > 0.5 else "ISSUE"),
        })
    return pd.DataFrame(results)

In [ ]:
prompts = [
    "Merkez Bankası'nın faiz kararının ardından piyasalarda",
    "Türkiye ekonomisinde son çeyrekte büyüme",
    "Borsa İstanbul'da bugün en çok yükselen sektör",
]

display(run_generation_quality(MODELS, TOKENIZERS, prompts))
clear_gpu()

## 17. Calibration: ECE and Brier Score

### What is it?
Is the model's **confidence** proportional to its actual **accuracy**? When the model says "I'm 90% sure", is it actually correct 90% of the time?

- **ECE (Expected Calibration Error):** bin predictions by confidence and take the weighted average of `|accuracy - confidence|` across bins. 0 = perfect calibration.
- **Brier Score:** `(1 - P(correct token))²` averaged across predictions. Captures both accuracy and calibration.

### What do we expect to find?
- Well-calibrated model: ECE < 0.05
- Overconfident model: ECE > 0.15 — the model is wrong precisely where it feels sure. Causes problems after fine-tuning.

### How do we interpret it?
| ECE | Verdict |
|-----|---------|
| < 0.05 | Well calibrated |
| 0.05-0.15 | Reasonable |
| > 0.15 | Overconfident — post-training calibration may be needed |

In [ ]:
def compute_calibration(model, tokenizer, texts, num_bins=10):
    """Compute ECE and Brier Score."""
    all_confidences = []
    all_accuracies = []
    brier_scores = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
        ids = inputs["input_ids"][0]
        with torch.no_grad():
            logits = model(**inputs).logits[0]

        probs = F.softmax(logits[:-1], dim=-1)
        targets = ids[1:]

        confidences, predictions = torch.max(probs, dim=-1)
        correct = (predictions == targets).float()

        target_probs = probs[torch.arange(len(targets)), targets]
        brier = torch.mean((1.0 - target_probs) ** 2).item()
        brier_scores.append(brier)

        all_confidences.extend(confidences.cpu().float().tolist())
        all_accuracies.extend(correct.cpu().tolist())

    # ECE
    confs = np.array(all_confidences)
    accs = np.array(all_accuracies)
    ece = 0.0
    bins = np.linspace(0, 1, num_bins + 1)
    for i in range(num_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if mask.sum() > 0:
            ece += np.abs(accs[mask].mean() - confs[mask].mean()) * mask.mean()

    return {"ECE": round(ece, 4), "Brier Score": round(np.mean(brier_scores), 4)}

In [ ]:
cal_texts = [
    "Merkez Bankası faiz kararı piyasalarda dalgalanmaya neden oldu.",
    "Borsa İstanbul günü yüzde iki artışla kapattı.",
    "Enflasyon oranı beklentilerin üzerinde açıklandı.",
    "Şirketin yıllık geliri bir önceki yıla göre yüzde on beş arttı.",
    "Dolar kuru son bir ayda yüzde beş değer kazandı.",
]

results = []
for name in MODELS:
    cal = compute_calibration(MODELS[name], TOKENIZERS[name], cal_texts)
    cal["Model"] = name
    results.append(cal)
display(pd.DataFrame(results)[["Model", "ECE", "Brier Score"]])
clear_gpu()

## 18. Paraphrase Consistency (Knowledge Robustness)

### What is it?
When the same question is asked in different forms, does the model give **the same answer**? Inconsistency means the knowledge is **fragile** — different answers across small prompt variations = the knowledge is shallow.

**Ref:** Elazar et al. (2021) "Measuring and Improving Consistency in PLMs"

### What do we expect to find?
- Good model: same answer (or same log-prob ranking) across 3 different formulations.
- Bad model: different answer per formulation → knowledge is not robust.

### How do we interpret it?
- **Consistent (3/3):** the knowledge is solidly encoded.
- **Partially consistent (2/3):** sensitive to formulation — strong on some patterns, weak on others.
- **Inconsistent (1/3 or 0/3):** knowledge is fragile — likely to cause issues during fine-tuning.

In [ ]:
def test_paraphrase_consistency(model, tokenizer, prompt_variants, choices, model_name=""):
    """Does the model give a consistent answer across paraphrases of the same question?"""
    winners = []
    details = []

    for prompt in prompt_variants:
        best_choice = None
        best_score = -float('inf')
        for choice in choices:
            lp = get_logprob(model, tokenizer, prompt, choice)
            if lp > best_score:
                best_score = lp
                best_choice = choice
        winners.append(best_choice)
        details.append({"Prompt": prompt[:50], "Winner": best_choice, "Score": round(best_score, 3)})

    unique = len(set(winners))
    consistent = unique == 1

    df = pd.DataFrame(details)
    df["Consistent"] = "PASS" if consistent else "FAIL"
    return df


paraphrase_tests = [
    {
        "variants": [
            "Merkez Bankası faiz oranını",
            "TCMB politika faizini",
            "Türkiye Cumhuriyet Merkez Bankası gösterge faizini",
        ],
        "choices": [" artırdı.", " düşürdü.", " sabit bıraktı."],
        "topic": "TCMB rate decision"
    },
    {
        "variants": [
            "Enflasyon yükselince tüketicinin alım gücü",
            "Fiyatlar genel düzeyi artınca halkın satın alma gücü",
            "Yüksek enflasyon ortamında vatandaşın alım gücü",
        ],
        "choices": [" düşer.", " artar.", " değişmez."],
        "topic": "Effect of inflation"
    },
]

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    for test in paraphrase_tests:
        print(f"\n  Topic: {test['topic']}")
        display(test_paraphrase_consistency(
            MODELS[name], TOKENIZERS[name],
            test["variants"], test["choices"], name
        ))
clear_gpu()